# Titanic Survival and Credit Card Fraud Detection

This notebook solves two binary classification assignments with PyTorch:

- Titanic survival prediction with a linear baseline and an MLP.
- Credit-card fraud detection with weighted loss, an MLP, and sampling strategies.

Local file note: `test.csv` is the credit-card fraud dataset. The local `train.csv` is checked below because it has a `.csv` extension but appears to be a Word document containing the assignment prompt, not Titanic rows. The Titanic code is complete and will run after replacing `train.csv` with a real Titanic training table containing `Survived`, `Age`, `Sex`, `Pclass`, `SibSp`, `Parch`, `Fare`, and `Embarked`.

## 1. Verify Environment and File Paths

Run the setup cells first. They confirm the local files, install missing packages if needed, and make later results reproducible.

In [ ]:
# Optional dependency setup. Run this cell if imports fail in your selected kernel.
import importlib.util
import subprocess
import sys

required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'matplotlib': 'matplotlib',
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('All required core packages are available.')

if importlib.util.find_spec('imblearn') is None:
    print('Optional package imbalanced-learn is not installed. SMOTE cell will be skipped unless you install it.')

In [ ]:
from pathlib import Path
import os
import random
import zipfile

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import resample

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR = Path.cwd()
TITANIC_PATH = BASE_DIR / 'train.csv'
CREDIT_PATH = BASE_DIR / 'test.csv'

print('Working directory:', BASE_DIR)
print('Device:', DEVICE)
for path in [TITANIC_PATH, CREDIT_PATH]:
    if path.exists():
        stat = path.stat()
        print(f'{path.name}: {stat.st_size:,} bytes, modified {pd.Timestamp(stat.st_mtime, unit="s")}')
    else:
        print(f'{path.name}: missing')

## 2. Inspect File Signatures and Metadata

The filenames are misleading, so this cell checks the first bytes before loading data.

In [ ]:
def first_bytes(path: Path, n: int = 16) -> bytes:
    with path.open('rb') as file:
        return file.read(n)

for path in [TITANIC_PATH, CREDIT_PATH]:
    if path.exists():
        signature = first_bytes(path)
        print(f'{path.name}: {signature!r} | hex={signature.hex("-")}')
        print('  zip container:', zipfile.is_zipfile(path))

## 3. Load Titanic Data Defensively

This loader tries CSV, Excel, and ZIP/Office metadata. If `train.csv` is the assignment document rather than a data table, the Titanic modeling cells will skip gracefully and tell you what to replace.

In [ ]:
def detect_office_package(path: Path) -> str | None:
    if not zipfile.is_zipfile(path):
        return None
    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())
    if 'word/document.xml' in names:
        return 'word-document'
    if 'xl/workbook.xml' in names:
        return 'excel-workbook'
    return 'zip-package'


def load_titanic(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f'{path.name} not found.')
        return None

    package_type = detect_office_package(path)
    if package_type == 'word-document':
        print(
            f'{path.name} is a Word document package containing the assignment prompt, '
            'not Titanic passenger rows. Replace it with a real Titanic train.csv '
            'before running the Titanic model cells.'
        )
        return None

    readers = []
    if package_type == 'excel-workbook':
        readers.append(lambda: pd.read_excel(path))
    readers.extend([
        lambda: pd.read_csv(path),
        lambda: pd.read_excel(path),
    ])

    last_error = None
    for reader in readers:
        try:
            data = reader()
            if {'Survived', 'Age', 'Sex', 'Pclass', 'SibSp', 'Parch', 'Fare', 'Embarked'}.issubset(data.columns):
                print('Loaded Titanic data:', data.shape)
                return data
            print('Loaded a table, but it does not contain the expected Titanic columns:', list(data.columns))
            return None
        except Exception as error:
            last_error = error
    print('Could not load Titanic data:', last_error)
    return None

titanic_df = load_titanic(TITANIC_PATH)
if titanic_df is not None:
    display(titanic_df.head())

## 4. Shared PyTorch Modeling Utilities

Both problems are binary classification, so one set of model and metric helpers is enough.

In [ ]:
class LogisticRegressionTorch(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)


class BinaryMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


def make_loader(X, y, batch_size=256, shuffle=True):
    X_tensor = torch.tensor(np.asarray(X), dtype=torch.float32)
    y_tensor = torch.tensor(np.asarray(y), dtype=torch.float32)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)


def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, pos_weight=None):
    model = model.to(DEVICE)
    if pos_weight is not None:
        pos_weight = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(DEVICE)
                batch_y = batch_y.to(DEVICE)
                val_losses.append(criterion(model(batch_X), batch_y).item())

        history.append({'epoch': epoch, 'train_loss': np.mean(train_losses), 'val_loss': np.mean(val_losses)})
        if epoch == 1 or epoch == epochs or epoch % max(1, epochs // 5) == 0:
            print(f'Epoch {epoch:03d} | train_loss={history[-1]["train_loss"]:.4f} | val_loss={history[-1]["val_loss"]:.4f}')
    return model, pd.DataFrame(history)


def predict_probabilities(model, X, batch_size=4096):
    model.eval()
    probabilities = []
    loader = DataLoader(torch.tensor(np.asarray(X), dtype=torch.float32), batch_size=batch_size)
    with torch.no_grad():
        for batch_X in loader:
            logits = model(batch_X.to(DEVICE))
            probabilities.extend(torch.sigmoid(logits).cpu().numpy())
    return np.asarray(probabilities)


def classification_metrics(y_true, y_prob, threshold=0.5, include_auc=False):
    y_pred = (y_prob >= threshold).astype(int)
    result = {
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    if include_auc:
        result['pr_auc'] = average_precision_score(y_true, y_prob)
        result['roc_auc'] = roc_auc_score(y_true, y_prob)
    return result


def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_values = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    best_index = int(np.nanargmax(f1_values))
    if best_index >= len(thresholds):
        return 0.5
    return float(thresholds[best_index])

# Part A: Titanic Survival Prediction

This section uses `Age`, `Sex`, `Pclass`, `SibSp`, `Parch`, `Fare`, and `Embarked`, then adds `FamilySize`, `IsAlone`, and optional `Title` extracted from `Name`.

In [ ]:
def add_titanic_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['FamilySize'] = data['SibSp'].fillna(0) + data['Parch'].fillna(0) + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    if 'Name' in data.columns:
        data['Title'] = data['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).fillna('Unknown')
        rare_titles = data['Title'].value_counts()[lambda counts: counts < 10].index
        data['Title'] = data['Title'].replace(rare_titles, 'Rare')
    return data


def prepare_titanic_arrays(data: pd.DataFrame):
    data = add_titanic_features(data)
    target = data['Survived'].astype(int)

    numeric_features = ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']
    categorical_features = ['Sex', 'Pclass', 'Embarked', 'IsAlone']
    if 'Title' in data.columns:
        categorical_features.append('Title')

    X_train_df, X_val_df, y_train, y_val = train_test_split(
        data[numeric_features + categorical_features],
        target,
        test_size=0.2,
        stratify=target,
        random_state=SEED,
    )

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ])

    X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
    X_val = preprocessor.transform(X_val_df).astype(np.float32)
    return X_train, X_val, y_train.to_numpy(), y_val.to_numpy(), preprocessor

if titanic_df is not None:
    X_titanic_train, X_titanic_val, y_titanic_train, y_titanic_val, titanic_preprocessor = prepare_titanic_arrays(titanic_df)
    print('Titanic train matrix:', X_titanic_train.shape)
    print('Titanic validation matrix:', X_titanic_val.shape)
else:
    print('Titanic modeling skipped until train.csv is replaced with a real Titanic data table.')

In [ ]:
titanic_results = []

if titanic_df is not None:
    train_loader = make_loader(X_titanic_train, y_titanic_train, batch_size=64)
    val_loader = make_loader(X_titanic_val, y_titanic_val, batch_size=256, shuffle=False)

    titanic_linear = LogisticRegressionTorch(X_titanic_train.shape[1])
    titanic_linear, titanic_linear_history = train_model(
        titanic_linear,
        train_loader,
        val_loader,
        epochs=80,
        lr=1e-3,
    )
    linear_prob = predict_probabilities(titanic_linear, X_titanic_val)
    titanic_results.append({'model': 'Logistic Regression torch.nn.Linear', **classification_metrics(y_titanic_val, linear_prob)})

    titanic_mlp = BinaryMLP(X_titanic_train.shape[1], hidden_dims=(32, 16), dropout=0.15)
    titanic_mlp, titanic_mlp_history = train_model(
        titanic_mlp,
        train_loader,
        val_loader,
        epochs=100,
        lr=1e-3,
    )
    mlp_prob = predict_probabilities(titanic_mlp, X_titanic_val)
    titanic_results.append({'model': 'MLP ReLU hidden layers', **classification_metrics(y_titanic_val, mlp_prob)})

    titanic_metrics = pd.DataFrame(titanic_results).sort_values('f1', ascending=False)
    display(titanic_metrics)
else:
    titanic_metrics = pd.DataFrame()
    print('No Titanic metrics produced because local train.csv is not passenger data.')

## Titanic Feature Engineering Discussion

`FamilySize` and `IsAlone` often improve Titanic results because survival depended partly on group behavior during evacuation. Very large families can be harder to evacuate, while solo passengers lose the signal carried by sibling, spouse, parent, and child counts. `Title` extracted from names can help because it captures gender, age group, social class, and marital status in a compact categorical feature. These features can improve an MLP and a linear model, but with a small dataset they also increase overfitting risk, so validation metrics matter more than training loss.

# Part B: Credit Card Fraud Detection

The provided `test.csv` matches the common credit-card fraud schema: `Time`, `V1` through `V28`, `Amount`, and `Class`. Accuracy is not enough here because the fraud class is extremely rare, so the focus is Precision, Recall, F1-score, and PR-AUC.

In [ ]:
expected_credit_columns = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']

credit_df = pd.read_csv(CREDIT_PATH)
credit_df.columns = [column.strip().replace('"', '') for column in credit_df.columns]
missing_columns = sorted(set(expected_credit_columns) - set(credit_df.columns))
if missing_columns:
    raise ValueError(f'Missing expected credit-card columns: {missing_columns}')

credit_df = credit_df[expected_credit_columns].copy()
credit_df['Class'] = pd.to_numeric(credit_df['Class'], errors='coerce')
credit_df = credit_df.dropna(subset=['Class']).drop_duplicates()
credit_df['Class'] = credit_df['Class'].astype(int)

print('Credit-card data shape:', credit_df.shape)
display(credit_df.head())
display(pd.DataFrame({'missing_values': credit_df.isna().sum(), 'dtype': credit_df.dtypes.astype(str)}).head(35))

In [ ]:
class_counts = credit_df['Class'].value_counts().sort_index()
class_distribution = pd.DataFrame({
    'count': class_counts,
    'percentage': class_counts / len(credit_df) * 100,
})
display(class_distribution)

ax = class_counts.rename({0: 'Genuine', 1: 'Fraud'}).plot(kind='bar', color=['#4C78A8', '#F58518'])
ax.set_title('Credit Card Class Distribution')
ax.set_ylabel('Transactions')
plt.xticks(rotation=0)
plt.show()

In [ ]:
credit_features = [column for column in credit_df.columns if column != 'Class']
X_credit = credit_df[credit_features]
y_credit = credit_df['Class'].to_numpy()

X_credit_train_df, X_credit_val_df, y_credit_train, y_credit_val = train_test_split(
    X_credit,
    y_credit,
    test_size=0.2,
    stratify=y_credit,
    random_state=SEED,
)

credit_scaler = StandardScaler()
X_credit_train = credit_scaler.fit_transform(X_credit_train_df).astype(np.float32)
X_credit_val = credit_scaler.transform(X_credit_val_df).astype(np.float32)

negative_count = int((y_credit_train == 0).sum())
positive_count = int((y_credit_train == 1).sum())
credit_pos_weight = negative_count / max(positive_count, 1)

print('Train shape:', X_credit_train.shape, 'Validation shape:', X_credit_val.shape)
print('Training fraud ratio:', y_credit_train.mean())
print('Positive class weight for BCEWithLogitsLoss:', round(credit_pos_weight, 2))

## Weighted Linear Baseline and MLP

The linear baseline uses `torch.nn.Linear`, equivalent to logistic regression when paired with `BCEWithLogitsLoss`. The loss is weighted so fraud examples have more influence during training.

In [ ]:
credit_train_loader = make_loader(X_credit_train, y_credit_train, batch_size=2048)
credit_val_loader = make_loader(X_credit_val, y_credit_val, batch_size=4096, shuffle=False)
credit_results = []

credit_linear = LogisticRegressionTorch(X_credit_train.shape[1])
credit_linear, credit_linear_history = train_model(
    credit_linear,
    credit_train_loader,
    credit_val_loader,
    epochs=8,
    lr=1e-3,
    pos_weight=credit_pos_weight,
)
linear_prob = predict_probabilities(credit_linear, X_credit_val)
linear_threshold = best_f1_threshold(y_credit_val, linear_prob)
credit_results.append({
    'model': 'Weighted Logistic Regression torch.nn.Linear',
    **classification_metrics(y_credit_val, linear_prob, threshold=linear_threshold, include_auc=True),
})

credit_mlp = BinaryMLP(X_credit_train.shape[1], hidden_dims=(64, 32), dropout=0.25)
credit_mlp, credit_mlp_history = train_model(
    credit_mlp,
    credit_train_loader,
    credit_val_loader,
    epochs=8,
    lr=1e-3,
    pos_weight=credit_pos_weight,
)
mlp_prob = predict_probabilities(credit_mlp, X_credit_val)
mlp_threshold = best_f1_threshold(y_credit_val, mlp_prob)
credit_results.append({
    'model': 'Weighted MLP',
    **classification_metrics(y_credit_val, mlp_prob, threshold=mlp_threshold, include_auc=True),
})

weighted_metrics = pd.DataFrame(credit_results)
display(weighted_metrics.sort_values('f1', ascending=False))
print('Weighted linear confusion matrix:')
print(confusion_matrix(y_credit_val, (linear_prob >= linear_threshold).astype(int)))
print('Weighted MLP confusion matrix:')
print(confusion_matrix(y_credit_val, (mlp_prob >= mlp_threshold).astype(int)))

## Sampling Strategy Experiments

Undersampling reduces the majority class, oversampling duplicates minority examples, and SMOTE synthetically interpolates minority examples when `imbalanced-learn` is installed. Compare PR-AUC and recall rather than accuracy.

In [ ]:
def undersample_training_data(X, y):
    train = pd.DataFrame(X)
    train['target'] = y
    majority = train[train['target'] == 0]
    minority = train[train['target'] == 1]
    majority_down = resample(
        majority,
        replace=False,
        n_samples=len(minority),
        random_state=SEED,
    )
    sampled = pd.concat([majority_down, minority]).sample(frac=1, random_state=SEED)
    return sampled.drop(columns='target').to_numpy(np.float32), sampled['target'].to_numpy()


def oversample_training_data(X, y):
    train = pd.DataFrame(X)
    train['target'] = y
    majority = train[train['target'] == 0]
    minority = train[train['target'] == 1]
    minority_up = resample(
        minority,
        replace=True,
        n_samples=len(majority),
        random_state=SEED,
    )
    sampled = pd.concat([majority, minority_up]).sample(frac=1, random_state=SEED)
    return sampled.drop(columns='target').to_numpy(np.float32), sampled['target'].to_numpy()


def train_sampling_experiment(name, X_sample, y_sample, epochs=6):
    loader = make_loader(X_sample, y_sample, batch_size=2048)
    model = BinaryMLP(X_sample.shape[1], hidden_dims=(64, 32), dropout=0.25)
    model, _ = train_model(model, loader, credit_val_loader, epochs=epochs, lr=1e-3)
    probabilities = predict_probabilities(model, X_credit_val)
    threshold = best_f1_threshold(y_credit_val, probabilities)
    return {'model': name, **classification_metrics(y_credit_val, probabilities, threshold=threshold, include_auc=True)}

sampling_results = []
X_under, y_under = undersample_training_data(X_credit_train, y_credit_train)
sampling_results.append(train_sampling_experiment('MLP undersampling', X_under, y_under))

X_over, y_over = oversample_training_data(X_credit_train, y_credit_train)
sampling_results.append(train_sampling_experiment('MLP oversampling', X_over, y_over))

try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=SEED, k_neighbors=5)
    X_smote, y_smote = smote.fit_resample(X_credit_train, y_credit_train)
    sampling_results.append(train_sampling_experiment('MLP SMOTE', X_smote.astype(np.float32), y_smote, epochs=6))
except Exception as error:
    print('SMOTE skipped:', error)

sampling_metrics = pd.DataFrame(sampling_results)
all_credit_metrics = pd.concat([weighted_metrics, sampling_metrics], ignore_index=True)
display(all_credit_metrics.sort_values('pr_auc', ascending=False))

## Generate Predictions and Save Outputs

Because `test.csv` already contains labels in `Class`, this export writes validation-set predictions for review. In a competition-style unlabeled test file, apply the same scaler to that file and run `predict_probabilities` on it.

In [ ]:
best_row = all_credit_metrics.sort_values('pr_auc', ascending=False).iloc[0]
best_model_name = best_row['model']

# For export, use the weighted MLP when it is competitive and already trained.
# You can swap this to a sampling model by storing that model in the experiment function.
export_probabilities = mlp_prob
export_threshold = mlp_threshold
prediction_output = X_credit_val_df.copy()
prediction_output['actual_Class'] = y_credit_val
prediction_output['fraud_probability'] = export_probabilities
prediction_output['predicted_Class'] = (export_probabilities >= export_threshold).astype(int)

output_path = BASE_DIR / 'credit_card_validation_predictions.csv'
prediction_output.to_csv(output_path, index=False)
print('Saved validation predictions to:', output_path)
print('Best PR-AUC model in comparison table:', best_model_name)

## Credit Card Fraud Discussion

Accuracy is misleading for this dataset because a model can predict every transaction as genuine and still look excellent. Precision, recall, F1-score, and PR-AUC are better aligned with fraud detection. Recall answers how many fraud cases are caught, precision answers how many flagged transactions are truly fraud, F1 balances the two, and PR-AUC summarizes performance across thresholds under severe imbalance.

Weighted loss usually improves recall because fraud examples contribute more to the gradient. Undersampling can train quickly and may improve recall, but it discards many genuine examples. Oversampling keeps all genuine examples but duplicates fraud rows, which can overfit. SMOTE can help by creating synthetic minority samples, but it may create unrealistic fraud-like points in PCA space. The best model should be selected using validation PR-AUC and an operating threshold that matches the business cost of missed fraud versus false alarms.